# Linear Regression Variants — Visualised

Three PySpark linear-regression scripts, run side by side on `BigBoy.xlsx`:

| Script | Behaviour |
|---|---|
| `LinearRegression.py` | Train on full data (random 80/20 split), evaluate on the test split. |
| `LinearRegressionDateLimit.py` | *Intended*: filter to **Nov 2022 → Dec 2025** before training. *Actual*: `filtered_df` is computed but never used (bug) — behaves identically to vanilla LR. |
| `LinearRegressionDateLimitV2.py` | Train on full data, evaluate only on the Nov 2022 → Dec 2025 subset of the test split. |

| # | Section |
|---|---|
| 1 | Setup |
| 2 | Data Loading |
| 3 | Helper Functions |
| 4 | Variant A — `LinearRegression.py` |
| 5 | Variant B — `LinearRegressionDateLimit.py` (as-written + as-intended) |
| 6 | Variant C — `LinearRegressionDateLimitV2.py` |
| 7 | Side-by-Side Comparison |
| 8 | Critical Assessment |

---
## 1. Setup

In [ ]:
import os
import sys
import re

os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

spark = (
    SparkSession.builder
    .appName("LinearRegression_Visualised")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version :", spark.version)
print("Python used   :", sys.executable)

---
## 2. Data Loading

The original scripts read `BigChungus.xlsx` and use the columns `Unnamed: 0` (date) and `LABEL_Next_Month_Inflation` (target). The workspace only contains `BigBoy.xlsx`, which has `Date` and `CPI After One Month`. We alias those names so the rest of the pipeline mirrors the scripts exactly.

In [ ]:
EXCEL_PATH = "BigBoy.xlsx"
DATE_COL   = "Unnamed: 0"
LABEL_COL  = "LABEL_Next_Month_Inflation"

raw = pd.read_excel(EXCEL_PATH)
raw = raw.rename(columns={"Date": DATE_COL, "CPI After One Month": LABEL_COL})
raw[DATE_COL] = pd.to_datetime(raw[DATE_COL])
raw = raw.sort_values(DATE_COL).reset_index(drop=True)

print(f"Rows  : {len(raw):,}")
print(f"Range : {raw[DATE_COL].min().date()} → {raw[DATE_COL].max().date()}")
print("\nMissing-value share per column:")
print(raw.drop(columns=[DATE_COL]).isna().mean().round(3).to_string())

raw.head()

---
## 3. Helper Functions

Encapsulate the feature engineering, fitting, evaluation and 4-panel visualisation so each variant can be expressed in a few lines.

In [ ]:
def sanitize(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]", "_", name).strip("_")


def build_features(pdf: pd.DataFrame, date_col: str, label_col: str):
    """Replicate the scripts' feature engineering with Spark-safe column names.

    Adds Date_Unix / Year / Month, keeps every other non-date column as a feature,
    and returns sanitized feature/label names plus a readable mapping.
    """
    base = [c for c in pdf.columns if c not in (label_col, date_col)]
    feature_cols_raw = ["Date_Unix", "Year", "Month"] + base

    pdf = pdf.copy()
    pdf["Date_Unix"] = pdf[date_col].astype("int64") // 10**9
    pdf["Year"]      = pdf[date_col].dt.year
    pdf["Month"]     = pdf[date_col].dt.month

    rename_map   = {c: sanitize(c) for c in feature_cols_raw + [label_col]}
    feature_cols = [rename_map[c] for c in feature_cols_raw]
    pretty       = {rename_map[c]: c for c in feature_cols_raw}

    return pdf.rename(columns=rename_map), feature_cols, rename_map[label_col], pretty


def fit_lr(train_sdf, feature_cols, label_col):
    asm   = VectorAssembler(inputCols=feature_cols, outputCol="features")
    model = LinearRegression(featuresCol="features", labelCol=label_col).fit(asm.transform(train_sdf))
    return asm, model


def eval_metrics(model, asm, sdf, label_col):
    preds = model.transform(asm.transform(sdf))
    metrics = {}
    for m in ("rmse", "mae", "r2"):
        ev = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName=m)
        metrics[m] = ev.evaluate(preds)
    return metrics, preds.select(label_col, "prediction", "Date_Unix").toPandas()


def metrics_from_pandas(pdf, label_col):
    return {
        "rmse": mean_squared_error(pdf[label_col], pdf["prediction"]) ** 0.5,
        "mae":  mean_absolute_error(pdf[label_col], pdf["prediction"]),
        "r2":   r2_score(pdf[label_col], pdf["prediction"]),
    }


def visualise(preds_pd, label_col, coefs, feature_cols, intercept, title, metrics, pretty=None):
    """4-panel diagnostic plot for a fitted regression."""
    pdf = preds_pd.copy()
    pdf["Date"]     = pd.to_datetime(pdf["Date_Unix"], unit="s")
    pdf             = pdf.sort_values("Date").reset_index(drop=True)
    pdf["residual"] = pdf[label_col] - pdf["prediction"]

    fig = plt.figure(figsize=(13, 9))
    gs  = fig.add_gridspec(3, 2)

    ax = fig.add_subplot(gs[0, :])
    ax.plot(pdf["Date"], pdf[label_col],   label="Actual",    linewidth=1.6, color="steelblue")
    ax.plot(pdf["Date"], pdf["prediction"], label="Predicted", linewidth=1.6, linestyle="--", color="darkorange")
    ax.set_title(f"{title}  —  Predicted vs Actual over time  (n={len(pdf):,})")
    ax.set_xlabel("Date"); ax.set_ylabel(label_col); ax.legend()

    ax = fig.add_subplot(gs[1, 0])
    ax.scatter(pdf[label_col], pdf["prediction"], alpha=0.6, s=22, color="steelblue")
    lims = [pdf[[label_col, "prediction"]].min().min(), pdf[[label_col, "prediction"]].max().max()]
    ax.plot(lims, lims, "k--", linewidth=1)
    ax.set_xlabel("Actual"); ax.set_ylabel("Predicted"); ax.set_title("Predicted vs Actual scatter")
    ax.text(0.05, 0.95,
            f"R²  = {metrics['r2']:.3f}\nRMSE = {metrics['rmse']:.3f}\nMAE = {metrics['mae']:.3f}",
            transform=ax.transAxes, fontsize=10, va="top",
            bbox=dict(facecolor="white", edgecolor="lightgrey"))

    ax = fig.add_subplot(gs[1, 1])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.scatter(pdf["Date"], pdf["residual"], alpha=0.65, s=20, color="crimson")
    ax.set_xlabel("Date"); ax.set_ylabel("Residual"); ax.set_title("Residuals over time")

    ax = fig.add_subplot(gs[2, :])
    feature_labels = [pretty.get(c, c) if pretty else c for c in feature_cols]
    coef_df = pd.DataFrame({"Feature": feature_labels, "Coefficient": coefs}).sort_values("Coefficient")
    sns.barplot(data=coef_df, x="Coefficient", y="Feature", ax=ax, color="steelblue")
    ax.set_title(f"Coefficients (intercept = {intercept:.3f})")

    plt.tight_layout(); plt.show()
    return pdf

---
## 4. Variant A — `LinearRegression.py`

- Random 80/20 split on the **full** date range (1947 → 2026).
- No date filtering.
- Drops every row where any feature or the label is NaN. Because the Kalshi columns (`Avg Predicted CPI`, `Market Confidence Score`, `Trading Intensity`) are missing pre-2021, the surviving training set will be much smaller than 1,863 rows.

In [ ]:
pdf, FCOLS, LCOL, PRETTY = build_features(raw, DATE_COL, LABEL_COL)
KEEP = FCOLS + [LCOL]

clean_A   = pdf.dropna(subset=KEEP)
first_d   = pd.to_datetime(clean_A["Date_Unix"], unit="s").min().date()
last_d    = pd.to_datetime(clean_A["Date_Unix"], unit="s").max().date()
print(f"Surviving rows after dropna: {len(clean_A):,}  ({first_d} → {last_d})")

sdf_A          = spark.createDataFrame(clean_A[KEEP].astype("float64"))
train_A, test_A = sdf_A.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_A.count():,}   Test: {test_A.count():,}")

asm_A, model_A   = fit_lr(train_A, FCOLS, LCOL)
metrics_A, preds_A = eval_metrics(model_A, asm_A, test_A, LCOL)
print("\nVariant A metrics:", {k: round(v, 4) for k, v in metrics_A.items()})

_ = visualise(preds_A, LCOL, model_A.coefficients.toArray(), FCOLS, model_A.intercept,
              "Variant A — vanilla LR (random split)", metrics_A, PRETTY)

---
## 5. Variant B — `LinearRegressionDateLimit.py`

**Bug in the original script.** Lines 51–53 build `filtered_df = sprk_df_with_time_features.filter(...)`, but line 61 then drops NaNs from `sprk_df_with_time_features` (the *unfiltered* DataFrame), and `filtered_df` is never referenced again. The script therefore behaves **identically to Variant A**.

We run two things here:

1. **B-as-written** — reproduces the bug, so its metrics are exactly Variant A.
2. **B-as-intended** — what the file name implies: filter to Nov 2022 → Dec 2025 *before* training.

In [ ]:
START, END = "2022-11-01", "2025-12-31"

metrics_B_buggy = dict(metrics_A)
print("B-as-written (== Variant A):", {k: round(v, 4) for k, v in metrics_B_buggy.items()})

mask         = (raw[DATE_COL] >= START) & (raw[DATE_COL] <= END)
pdf_B, _, _, _ = build_features(raw[mask], DATE_COL, LABEL_COL)
clean_B       = pdf_B.dropna(subset=KEEP)
print(f"\nB-intended rows after filter+dropna: {len(clean_B):,}")

if len(clean_B) < 10:
    print("\n[!] Too few rows after filter+dropna to fit a meaningful model. Skipping B-intended.")
    metrics_B = {"rmse": np.nan, "mae": np.nan, "r2": np.nan}
else:
    sdf_B          = spark.createDataFrame(clean_B[KEEP].astype("float64"))
    train_B, test_B = sdf_B.randomSplit([0.8, 0.2], seed=42)
    print(f"Train: {train_B.count():,}   Test: {test_B.count():,}")

    asm_B, model_B    = fit_lr(train_B, FCOLS, LCOL)
    metrics_B, preds_B = eval_metrics(model_B, asm_B, test_B, LCOL)
    print("\nB-intended metrics:", {k: round(v, 4) for k, v in metrics_B.items()})

    _ = visualise(preds_B, LCOL, model_B.coefficients.toArray(), FCOLS, model_B.intercept,
                  "Variant B-intended — filter → train", metrics_B, PRETTY)

---
## 6. Variant C — `LinearRegressionDateLimitV2.py`

Train on the full random 80/20 split (same model as Variant A), then keep only the rows of the **test** prediction whose date falls in `[Nov 2022, Dec 2025]`. So we re-use `model_A` and post-filter.

In [ ]:
preds_test_full = (
    model_A.transform(asm_A.transform(test_A))
    .select(LCOL, "prediction", "Date_Unix")
    .toPandas()
)
preds_test_full["Date"] = pd.to_datetime(preds_test_full["Date_Unix"], unit="s")

mask_C  = (preds_test_full["Date"] >= START) & (preds_test_full["Date"] <= END)
preds_C = preds_test_full[mask_C].copy()
print(f"Test rows kept in [{START}, {END}]: {len(preds_C):,}  /  {len(preds_test_full):,}")

if len(preds_C) < 5:
    print("\n[!] Fewer than 5 test rows fall in the window — metrics will be unstable.")

metrics_C = metrics_from_pandas(preds_C, LCOL) if len(preds_C) >= 2 else {"rmse": np.nan, "mae": np.nan, "r2": np.nan}
print("\nVariant C metrics:", {k: round(v, 4) for k, v in metrics_C.items()})

if len(preds_C) >= 2:
    _ = visualise(preds_C, LCOL, model_A.coefficients.toArray(), FCOLS, model_A.intercept,
                  "Variant C — train all, evaluate on Nov 2022 → Dec 2025", metrics_C, PRETTY)

---
## 7. Side-by-Side Comparison

In [ ]:
summary = pd.DataFrame({
    "A — vanilla":              metrics_A,
    "B-as-written (== A)":      metrics_B_buggy,
    "B-intended (filter→train)": metrics_B,
    "C (filter→eval only)":      metrics_C,
}).T[["rmse", "mae", "r2"]].round(4)

print(summary.to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, m in zip(axes, ("rmse", "mae", "r2")):
    sns.barplot(x=summary.index, y=summary[m], ax=ax, color="steelblue")
    ax.set_title(m.upper()); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=20)
    if m == "r2":
        ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout(); plt.show()

---
## 8. Critical Assessment

**Short version: these models are not effective forecasters of next-month CPI.** Whatever R² they produce mostly reflects how strongly `CPI` already predicts `CPI After One Month`, not any genuine forecasting skill.

### What is fundamentally wrong, in all three variants

1. **Trivial autocorrelation drives the apparent fit.** `CPI` (current month) is included as a feature, and the label is `CPI After One Month`. CPI changes by about 0.3 on a value of ~250, so any model that simply outputs `CPI_next ≈ CPI_current` already gets RMSE ≈ 0.3 and R² close to 1.0. A linear regression that effectively just memorises that identity will look fantastic and add no information.

2. **Random 80/20 split on a time series is leakage.** With dates shuffled into the train and test sets, the model is interpolating between known points rather than forecasting. A fair benchmark for next-month forecasting is a **chronological** split (train on past, test on future), as the GBT notebook uses.

3. **Including `Year`, `Month`, `Date_Unix` as raw numeric features is a poor fit for linear regression.**
   - `Date_Unix` is monotonically increasing and highly collinear with `Year`. The model will load most of the trend onto these and produce an enormous negative intercept, which doesn't generalise outside the train period.
   - `Month` enters as 1–12, which forces a *linear* effect of month-of-year. Real CPI seasonality is non-linear (peaks in spring, dips in autumn) and should be one-hot encoded or expressed as sin/cos pairs.

4. **`dropna` over all features collapses the training set.** The Kalshi columns are missing for nearly all pre-2021 rows, so any row outside the Kalshi window is dropped. The 1,863-row dataset shrinks dramatically (often to a few dozen rows), and the model is then fit on a tiny, recent, non-representative window.

5. **No imputation, no scaling, no regularisation.** With ~10 features and very few rows after dropna, plain OLS will overfit. `LinearRegression` in Spark supports `regParam` and `elasticNetParam` — neither is used.

### Variant-specific issues

**Variant A — `LinearRegression.py`.** Random split + `CPI` as a feature is a recipe for an artificially high R² that says nothing about forecasting skill.

**Variant B — `LinearRegressionDateLimit.py`.** Has the silent bug noted above: `filtered_df` is created and never used, so the script secretly behaves like Variant A. The *intended* version (filter → train) restricts to ~38 months of data; with ~12 features that is firmly into overfitting territory. Worse, the random 80/20 split inside such a tiny window means the test set is ~7 months of essentially adjacent observations — the metric is almost meaningless.

**Variant C — `LinearRegressionDateLimitV2.py`.** Trains on full data, keeps only test predictions inside `[Nov 2022, Dec 2025]`. Two problems:
- After random splitting, only a handful of test rows fall in the window, so the metric is high-variance.
- The reported R² is over a narrow date window where actual CPI moved a lot (post-COVID inflation). With `CPI` as a feature, the model can still get a decent number for the wrong reasons.

### How effective are they, really?

Even if these scripts print R² numbers in the 0.95–0.99 range, that performance is **not** evidence of a useful CPI forecaster. To meaningfully claim skill, the same evaluation would need:

- A **chronological** train/test split (train ≤ some cut-off, test strictly after).
- A **baseline** comparison — most importantly `CPI_next ≈ CPI_current` (random walk) and a simple AR(1) on Δ CPI. Anything that doesn't beat the random walk by a clear margin has zero forecasting value.
- The label re-expressed as **Δ CPI** (or year-over-year inflation), which is what the column name `LABEL_Next_Month_Inflation` actually implies. Predicting the *change* removes the trivial autocorrelation and forces the model to use the macro features.
- Proper handling of the Kalshi columns: either restrict the entire experiment to the Kalshi window, or drop those features for the historical comparison.
- Time-series cross-validation (rolling/expanding window) instead of a single random split.

Until those changes are made, **all three scripts are essentially the same model dressed up three different ways**, and their numbers should be treated as a sanity check that the code runs — not as evidence of a working CPI forecaster.

In [ ]:
spark.stop()